# DAY 10 -- The Wrapper Patterns (Decorators)

### MC10.1 : The Manual Wrapper


> Goal : Write a function `wrapper(func)` that runs code before/after. Apply it manually: `new_func = wrapper(old_func)`.


In [1]:
# Write a wrapper function that runs code before/after. Apply it manually
def wrapper(func):
    def inner():
        print('before')
        func()
        print('after')
    return inner

def old_func():
    print('work')

new_func = wrapper(old_func)
new_func()


before
work
after


> **Deep Dive:** This demystifies the `@` syntax. A decorator is simply a function that takes a function and returns a function.


---


### MC10.2 : The Syntax Sugar


> Goal : Apply the wrapper using the `@decorator` syntax.


In [2]:
# Apply the wrapper using the `@decorator` syntax.
def decorator(func):
    def inner():
        print('before')
        func()
        print('after')
    return inner

@decorator
def greet():
    print('hi')

greet()


before
hi
after


> **Deep Dive:** The `@` symbol is *Syntax Sugar*. At compile time, Python reads this and automatically performs the reassignment:
> `func = decorator(func)`.


---


### MC10.3 : The Args Problem


> Goal : Try to decorate a function that takes arguments (`add(a, b)`) with a wrapper that takes none.


In [3]:
# Try to decorate a function that takes arguments (`add(a, b)`) with a wrapper that takes none.
def wrapper(func):
    def inner():
        return func()
    return inner

@wrapper
def add(a, b):
    return a + b

add(1, 2)


TypeError: wrapper.<locals>.inner() takes 0 positional arguments but 2 were given

> **Deep Dive:** It crashes. The inner wrapper function **MUST** accept `*args` and `**kwargs` to be compatible with any target function signature.


---


### MC10.4 : The Return Value Thief


> Goal : Write a decorator that forgets to return `func(*args)`. Print the result of the decorated function.


In [4]:
# Write a decorator that forgets to return `func(*args)`. Print the result of the decorated function.
def wrapper(func):
    def inner(*args, **kwargs):
        func(*args, **kwargs)
    return inner

@wrapper
def add(a, b):
    return a + b

print(add(1, 2))


None


> **Deep Dive:** It prints `None`. A wrapper must explicitly capture and return the result of the original function, otherwise the data is lost inside the wrapper scope.


---


### MC10.5 : The Timer (Performance)


> Goal : Create a `@timer` decorator that prints execution time using `time.time()`.


In [5]:
# Create a `@timer` decorator that prints execution time using `time.time()`.
import time

def timer(func):
    def inner(*args, **kwargs):
        t0 = time.time()
        result = func(*args, **kwargs)
        print(time.time() - t0)
        return result
    return inner

@timer
def work():
    for _ in range(100000):
        pass

work()


0.013074874877929688


> **Deep Dive:** This is AOP (Aspect-Oriented Programming). We inject timing logic into the function without modifying the function’s actual code.


---


### MC10.6 : The Authenticator (Guard)


> Goal : Create `@admin_required`. If global `USER != 'admin'`, raise `PermissionError`.


In [6]:
# Create `@admin_required`. If global `USER != 'admin'`, raise `PermissionError`.
USER = 'guest'

def admin_required(func):
    def inner(*args, **kwargs):
        if USER != 'admin':
            raise PermissionError
        return func(*args, **kwargs)
    return inner

@admin_required
def secret():
    return 'ok'

secret()


PermissionError: 

In [12]:
USER = 'admin'

secret() # Now it works because USER is 'admin'.

'ok'

> **Deep Dive:** The decorator acts as a gatekeeper. It executes *before* the sensitive function is entered. If the check fails, the stack frame for the target function is never even created.


---


### MC10.7 : The Memorizer (Cache)


> Goal : Write a `@cache` decorator that stores results of expensive function calls in a dictionary.


In [7]:
# Write a `@cache` decorator that stores results of expensive function calls in a dictionary.
def cache(func):
    memo = {}
    def inner(n):
        if n in memo:
            return memo[n]
        memo[n] = func(n)
        return memo[n]
    return inner

@cache
def fib(n):
    return n if n < 2 else fib(n-1) + fib(n-2)

print(fib(10))


55


> **Deep Dive:** If `func(5)` is called, check the dict. If key `5` exists, return it instantly (`O(1)`).
> If not, run the function and save the result. This optimizes recursion (e.g., Fibonacci).


---


### MC10.8 : The Metadata Fix


> Goal : Print `func.__name__` of a decorated function.


In [8]:
# Print `func.__name__` of a decorated function.
import functools

def deco(func):
    @functools.wraps(func)
    def inner(*args, **kwargs):
        return func(*args, **kwargs)
    return inner

@deco
def hello():
    pass

print(hello.__name__)


hello


> **Deep Dive:** It prints `"wrapper"`, not the original name. This confuses debuggers.
> *Fix:*
> Use `@functools.wraps(func)` on the wrapper to copy the original metadata (name, docstring) to the new function.


---


### MC10.9 : The Stacked Decorator


> Goal : Apply two decorators: `@bold` and `@italic` to a string-returning function.


In [9]:
# Apply two decorators: `@bold` and `@italic` to a string-returning function.
def bold(func):
    def inner():
        return f'<b>{func()}</b>'
    return inner

def italic(func):
    def inner():
        return f'<i>{func()}</i>'
    return inner

@bold
@italic
def text():
    return 'hi'

print(text())


<b><i>hi</i></b>


> **Deep Dive:** Decorators stack from **bottom to top** (Inner → Outer).
> `@bold @italic` becomes `bold(italic(func()))`.
> Order matters.


---


### MC10.10 : Decorators with Arguments


> Goal : Create a decorator that accepts a setting: `@repeat(times=3)`.


In [10]:
# Create a decorator that accepts a setting: `@repeat(times=3)`.
def repeat(times):
    def deco(func):
        def inner(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(times)]
        return inner
    return deco

@repeat(times=3)
def ping():
    return 'ping'

print(ping())


['ping', 'ping', 'ping']


> **Deep Dive:** This requires **Three Levels of Nested Functions**:
> 1. The **Factory** (accepts `times`)
> 2. The **Decorator** (accepts `func`)
> 3. The **Wrapper** (accepts `*args`)


---
